# Experimentación EXP-01, EXP-02 y EXP-03 con los datos de la Pastelería y Panadería DEREK

**Base de Datos Avanzadas y Big Data · Proyecto final 2026-2 · Capítulo IV del informe (diseño del experimento) en Google Colab con Spark SQL**

El presente laboratorio documenta el desarrollo de los experimentos EXP-01, EXP-02 y EXP-03, correspondientes al diseño experimental establecido en el Capítulo IV del proyecto de la Pastelería y Panadería DEREK.

La experimentación se desarrolló utilizando Google Colab, Apache Spark SQL 4.0.1 y los conjuntos de datos de ventas correspondientes a los volúmenes V1, V2 y V3.

Se evaluaron dos estructuras de almacenamiento (3FN y estrella), dos patrones de procesamiento (ETL y ELT) y el comportamiento de las consultas y cargas de datos frente al incremento del volumen de registros.

El procedimiento comprendió la configuración del entorno, validación de los datos, construcción de las estructuras, ejecución de las pruebas, análisis estadístico, evaluación de las reglas de decisión y elaboración de conclusiones.

Los resultados se documentaron mediante registros de ejecución, tablas consolidadas, gráficos y archivos de evidencia.


## 0. Diseño y protocolo experimental

La experimentación se desarrolló tomando como referencia el diseño establecido en el Capítulo IV del informe, en las secciones 4.1 a 4.5. El equipo responsable corresponde a los integrantes consignados en la portada del proyecto. El protocolo define las condiciones de ejecución, las variables evaluadas y los criterios utilizados para analizar los resultados de los experimentos EXP-01 y EXP-02.

- **Problema de ingeniería y contexto de decisión:** el propietario de la Pastelería y Panadería DEREK requiere información sobre los 10 productos más vendidos de cada mes para apoyar las decisiones relacionadas con la producción y las compras. Para ello, se utiliza la consulta Q1 como referencia de evaluación.
- **Pregunta experimental (EXP-01):** ¿en qué medida la estructura de almacenamiento (3FN frente a estrella) y el volumen de registros (V1, V2 y V3) modifican la latencia p95, el throughput y la exactitud de Q1, manteniendo constantes la concurrencia y la política de caché?
- **Pregunta experimental (EXP-02):** ¿cómo varía el rendimiento de los patrones de carga ETL y ELT al incrementar el volumen de datos y qué comportamiento presentan respecto a la integridad, reconciliación e idempotencia?
- **Objetivo de la experimentación:** evaluar el comportamiento de las estructuras 3FN y estrella mediante la consulta Q1, utilizando los volúmenes V1 (100 000 registros), V2 (500 000 registros) y V3 (1 000 000 de registros). Se consideran como métricas la latencia p95, el throughput de consultas, la exactitud de los resultados y el rendimiento de carga de los patrones ETL y ELT. La evaluación se realiza mediante cinco repeticiones por tratamiento y las reglas de decisión establecidas para cada experimento.

### Hipótesis evaluadas

- **H1. Latencia:** el esquema en estrella reduce al menos un 30 % la latencia p95 de Q1 frente a la estructura 3FN en cada volumen, considerando la mediana de cinco repeticiones. Además, se plantea que la diferencia absoluta de latencia aumenta conforme se incrementa el volumen de datos. **H0:** no se alcanza la reducción establecida en alguno de los volúmenes o la diferencia absoluta no aumenta.
- **H2. Exactitud:** ambas estructuras devuelven el mismo Top-10 mensual que la referencia calculada mediante pandas, con una exactitud del 100 %. **H0:** se presenta al menos una diferencia en los resultados o en el orden de las posiciones.
- **H3. Rendimiento de carga:** el throughput de carga en V3 no disminuye más del 30 % respecto de V1, tanto para ETL como para ELT. **H0:** la disminución supera el 30 % en alguno de los patrones.
- **H4. Calidad e idempotencia:** la tasa de rechazo es inferior al 1 % y una segunda ejecución sobre los mismos datos no incorpora filas nuevas. **H0:** la tasa de rechazo es igual o superior al 1 % o la segunda ejecución incorpora nuevas filas.

### Variables y condiciones de control

Las variables independientes consideradas son la estructura de almacenamiento (A = 3FN y B = estrella), el patrón de carga (ETL y ELT) y el volumen de datos (V1, V2 y V3). Las variables dependientes son la latencia p95 (ms), el throughput de consultas (consultas/s), la exactitud (%), la duración de carga, el throughput de carga (filas/s), la tasa de rechazo (%), la reconciliación y la idempotencia.

Como condiciones de control se establecieron la consulta Q1, los datos utilizados, el entorno de ejecución, la concurrencia de cinco sesiones con veinte consultas por sesión, el calentamiento de diez consultas, la semilla 2026, la configuración de Spark y el orden de ejecución definido en la Tabla 4.7 del informe.

### Criterios de decisión

**EXP-01. Comparación entre 3FN y estrella**

Se considera que la estructura en estrella presenta una mejora frente a 3FN cuando cumple simultáneamente las siguientes condiciones:

1. La mediana de la latencia p95 es, como mínimo, un 30 % menor.
2. Los rangos mínimo-máximo de latencia de ambas estructuras no presentan solapamiento.
3. La exactitud de los resultados de Q1 es del 100 %.
4. La latencia p95 de la estructura en estrella es igual o inferior a 2 000 ms en V3.

**EXP-02. Evaluación de ETL y ELT**

Se considera que un patrón de carga cumple el criterio de escalabilidad cuando, utilizando la mediana de cinco repeticiones, satisface las siguientes condiciones:

1. El throughput de carga en V3 representa al menos el 70 % del throughput obtenido en V1.
2. La validación finaliza sin errores y la reconciliación presenta una diferencia igual al número de registros rechazados.
3. La tasa de rechazo es inferior al 1 %.
4. La segunda ejecución sobre los mismos datos no incorpora filas nuevas.

Cuando el coeficiente de variación entre repeticiones supera el 10 %, las mediciones correspondientes deben repetirse antes de emitir una conclusión sobre el tratamiento evaluado.

### Condiciones, recursos y limitaciones

La experimentación utiliza conjuntos de datos sintéticos, un único entorno de ejecución y cinco repeticiones por tratamiento. El procesamiento en Spark se realiza en modo local mediante `local[2]`, sin utilización de GPU. Estas condiciones permiten comparar los tratamientos dentro del mismo entorno, pero limitan la generalización de los resultados a otros equipos o configuraciones distribuidas.

Entre las principales limitaciones se consideran la baja cardinalidad de algunas dimensiones, la dependencia de un único equipo, la variabilidad propia del entorno de Colab y la persistencia de la caché de la JVM y del sistema operativo. Por ello, los resultados se interpretan dentro de las condiciones específicas en las que se ejecutó cada experimento.

Los recursos utilizados comprenden Google Colab, Apache Spark 4.0.1, Java 21, archivos Parquet y los conjuntos de datos preparados para los tres volúmenes experimentales.


| # | Capítulo IV (SQL Server) | Esta ejecución (Colab + Spark) | Efecto sobre la validez |
|---|---|---|---|
| 1 | SQL Server 2025 Express, equipo Lenovo i5, 8 GB | Spark 4.0.1 `local[2]` en Colab (CPU, sin GPU) | Los tiempos no son comparables con SQL Server ni con otro equipo |
| 2 | Reinicio del servicio antes de cada ejecución | `clearCache()` y relectura de Parquet; la JVM y la caché del sistema operativo no se reinician | La política de caché es idéntica entre tratamientos, pero no es lectura en frío |
| 3 | V1 = 100 000, V2 = 500 000, V3 = 1 000 000 | V1 = *mediano* (100 000), V3 = *grande* (1 000 000); **V2 se deriva** de *grande* con `sample(n=500000, random_state=2026)`; *pequeño* (1 000) solo sirve para la prueba de humo | V2 comparte líneas con V3; *grande* ya es sintético (LEEME) |
| 4 | Tablas con productos, SCD tipo 2 y compras | Solo hay líneas de venta: `dim_producto` tiene una versión por producto y el throughput se mide solo sobre ventas | No se evalúa SCD ni la carga de compras |
| 5 | Índices de claves primarias y UNIQUE | Parquet no usa índices: ninguna estructura recibe ajustes específicos | Se mantiene la regla de no favorecer a ninguna estructura |
| 6 | `Measure-Command` y `etl.ejecucion` | `time.perf_counter()` y registros JSONL | Resolución de milisegundos |
| 7 | Orden aleatorizado de la Tabla 4.7 | Se usa literalmente la Tabla 4.7 | Sin cambio |


## 1 Instalación en Google Colab

Se ejecuta las dos celdas una vez. Spark queda fijado en 4.0.1 para usar la misma API; no se presenta como la versión más reciente. Se requiere Python 3.9 o superior y Java 17 o 21; la segunda celda instala Java 17 si hace falta. Si Colab solicita reiniciar, reinicia y vuelve a ejecutar desde esta etapa. Sin GPU.

In [1]:
%pip -q install pyspark==4.0.1 "numpy>=1.24,<3" "pandas>=2.0,<3" "scipy>=1.11,<2" "matplotlib>=3.9,<4"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import shutil, subprocess, re
def java_mayor():
    if shutil.which("java") is None:
        return None
    t = subprocess.run(["java", "-version"], capture_output=True, text=True)
    m = re.search(r'version "(\d+)', t.stderr + t.stdout)
    return int(m.group(1)) if m else None

if java_mayor() not in (17, 21):
    subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless", shell=True, check=True)
print("Java mayor:", java_mayor())

Java mayor: 21



## 2. Parámetros y entorno experimental

Se configuró el entorno en modo completo (`MODO = "completo"`), estableciendo tres volúmenes de datos (V1, V2 y V3) y cinco repeticiones por tratamiento. Para EXP-01 se definieron cinco sesiones concurrentes, con veinte consultas por sesión, equivalentes a 100 consultas por repetición. En EXP-02, cada repetición comprende una ejecución completa del proceso de carga del Data Mart.

El modo de prueba (`MODO = "prueba"`) se utilizó únicamente para verificar el funcionamiento del código, sin considerar sus mediciones en el análisis final.


In [3]:
import os, sys, re, json, time, platform, subprocess, hashlib, shutil, math, zipfile
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import matplotlib.pyplot as plt
import pyspark
from pyspark import StorageLevel
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import (StructType, StructField, LongType, IntegerType,
                               DoubleType, StringType, TimestampType)

MODO = "completo"          # "prueba" para depurar con el volumen pequeño
USAR_DRIVE = False         # True: guarda las evidencias en Google Drive
RUN_ID_REANUDAR = None     # p. ej. "20261124T150000_000000Z" para reanudar

SEED = 2026
UMBRAL_REDUCCION = 0.30    # H1: B reduce >= 30 % la p95
UMBRAL_P95_MS = 2000       # RNF-01
UMBRAL_THROUGHPUT = 0.70   # H3
UMBRAL_RECHAZO_PCT = 1.0   # RNF-07
CV_MAX = 10.0
ALFA = 0.05
TOP_N = 10

if MODO == "prueba":
    VOLUMENES = {"V1": ("pequeno", 1_000)}
    REPS, SESIONES, CONSULTAS_SESION, CALENTAMIENTO = 2, 2, 5, 2
else:
    VOLUMENES = {"V1": ("mediano", 100_000), "V2": ("derivado_500k", 500_000), "V3": ("grande", 1_000_000)}
    REPS, SESIONES, CONSULTAS_SESION, CALENTAMIENTO = 5, 5, 20, 10

# Tabla 4.7 del informe (semilla 2026), usada literalmente.
ORDEN_VOLUMENES = ["V3", "V2", "V1"]
ORDEN_EXP02 = {
 "V3": "ETL1 ELT2 ETL3 ETL4 ELT1 ETL5 ELT3 ETL2 ELT5 ELT4".split(),
 "V2": "ETL3 ELT4 ETL2 ELT1 ELT2 ETL5 ETL1 ETL4 ELT3 ELT5".split(),
 "V1": "ELT4 ELT2 ETL5 ETL4 ETL2 ETL3 ELT1 ELT5 ETL1 ELT3".split()}
ORDEN_EXP01 = {
 "V3": "A5 A4 B3 B4 A1 A3 B5 A2 B2 B1".split(),
 "V2": "B3 A4 A2 B4 B5 B1 B2 A1 A5 A3".split(),
 "V1": "B1 A3 B5 A5 A4 A1 B4 B3 B2 A2".split()}
for v in ORDEN_EXP02:
    assert sorted(ORDEN_EXP02[v]) == sorted(f"{p}{i}" for p in ("ETL", "ELT") for i in range(1, 6))
    assert sorted(ORDEN_EXP01[v]) == sorted(f"{p}{i}" for p in "AB" for i in range(1, 6))

RUN_ID = RUN_ID_REANUDAR or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
BASE = Path.cwd()
if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive")
ROOT = BASE / ("evidencias_DEREK_" + RUN_ID)
for carpeta in ["datos", "tablas", "figuras", "planes", "logs/spark_events"]:
    (ROOT / carpeta).mkdir(parents=True, exist_ok=True)

java_text = subprocess.run(["java", "-version"], capture_output=True, text=True)
java_info = java_text.stderr + java_text.stdout
major = int(re.search(r'version "(\d+)', java_info).group(1))
assert major in (17, 21), "Usa Java 17 o 21 y reinicia el entorno."
os.environ["JAVA_HOME"] = str(Path(shutil.which("java")).resolve().parent.parent)
os.environ["PYSPARK_PYTHON"] = sys.executable

spark = (SparkSession.builder.master("local[2]")
    .appName("DEREK_Experimentacion")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", (ROOT / "logs/spark_events").as_uri())
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
assert spark.version == "4.0.1", "Reinicia el entorno para usar la versión fijada."

ram_gb = None
try:
    ram_gb = int(re.search(r"MemTotal:\s+(\d+)", open("/proc/meminfo").read()).group(1)) / 1e6
except Exception:
    pass
entorno = dict(run_id=RUN_ID, modo=MODO, semilla=SEED, volumenes=VOLUMENES, reps=REPS,
    sesiones=SESIONES, consultas_por_sesion=CONSULTAS_SESION, calentamiento=CALENTAMIENTO,
    python=sys.version, spark=spark.version, java=java_info, pandas=pd.__version__,
    numpy=np.__version__, scipy=scipy.__version__, sistema=platform.platform(),
    cpu_logicas=os.cpu_count(), ram_gb=ram_gb, disco_libre_gb=shutil.disk_usage(BASE).free / 1e9,
    master=spark.sparkContext.master, spark_conf=dict(spark.sparkContext.getConf().getAll()))
(ROOT / "entorno.json").write_text(json.dumps(entorno, indent=2, ensure_ascii=False, default=str))
print("MODO:", MODO, "| Evidencias:", ROOT)
print("Spark", spark.version, "|", spark.sparkContext.master, "| Java", major, "| RAM GB:", ram_gb)

MODO: completo | Evidencias: /content/evidencias_DEREK_20261125T032248_728595Z
Spark 4.0.1 | local[2] | Java 21 | RAM GB: 13.286932


## 3 Carga de los datos de la panadería y construcción de V1, V2 y V3

se sube los ZIP `Dataset_Panaderia_pequeno.zip`, `Dataset_Panaderia_mediano.zip` y `Dataset_Panaderia_grande.zip` (o déjalos en el directorio de trabajo). Cada ZIP trae `csv/<volumen>.csv`, el mismo contenido en Parquet, `calidad_<volumen>.csv` y `manifest.json` con los SHA-256. La celda verifica cada hash antes de usar los datos.

*Mediano* y *pequeño* son muestras de las 150 000 líneas de venta de la base `Panaderia`; *grande* es **sintético**: repite líneas reales al azar con IdDetalle nuevos (LEEME). V2 (500 000 líneas) se obtiene de *grande* con una muestra determinista de semilla 2026. Se verifica que su distribución mensual difiera menos de 5 % de la de V3, como pide el paso 3 del procedimiento (4.6).

In [4]:
ZIP_DIR = Path.cwd()
necesarios = ["pequeno"] if MODO == "prueba" else ["mediano", "grande"]
ZIPS = {n: ZIP_DIR / f"Dataset_Panaderia_{n}.zip" for n in ["pequeno", "mediano", "grande"]}
faltan = [n for n in necesarios if not ZIPS[n].exists()]
if faltan:
    try:
        from google.colab import files
        print("Sube:", [f"Dataset_Panaderia_{n}.zip" for n in faltan])
        files.upload()          # Colab guarda los archivos en el directorio actual
    except ImportError:
        pass
faltan = [n for n in necesarios if not ZIPS[n].exists()]
assert not faltan, f"Faltan los ZIP de: {faltan}"

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

FUENTE = ROOT / "datos" / "fuente"
hashes = []
for n, zp in ZIPS.items():
    if not zp.exists():
        continue
    with zipfile.ZipFile(zp) as z:
        z.extractall(FUENTE / n)
    manifest = json.loads((FUENTE / n / "manifest.json").read_text())
    for item in manifest:
        real = sha256(FUENTE / n / item["ruta"])
        assert real == item["sha256"], f"Hash distinto en {n}/{item['ruta']}"
        hashes.append(dict(volumen=n, ruta=item["ruta"], bytes=item["bytes"], sha256=real, coincide=True))
print("Hashes verificados:", len(hashes))

if "V2" in VOLUMENES:
    destino = FUENTE / "derivado_500k" / "csv" / "derivado_500k.csv"
    destino.parent.mkdir(parents=True, exist_ok=True)
    g = pd.read_csv(FUENTE / "grande/csv/grande.csv")
    v2 = g.sample(n=VOLUMENES["V2"][1], random_state=SEED).sort_values("IdDetalle")
    v2.to_csv(destino, index=False)
    m3 = pd.to_datetime(g.FechaVenta).dt.strftime("%Y-%m").value_counts(normalize=True)
    m2 = pd.to_datetime(v2.FechaVenta).dt.strftime("%Y-%m").value_counts(normalize=True)
    desvio = float(((m2 - m3).abs() / m3).max())
    assert desvio < 0.05, f"La distribución mensual de V2 difiere {100*desvio:.2f} % de V3"
    hashes.append(dict(volumen="derivado_500k", ruta="csv/derivado_500k.csv", bytes=destino.stat().st_size,
                       sha256=sha256(destino), coincide=None, metodo="sample(n=500000, random_state=2026) de grande", desvio_mensual_max_pct=100 * desvio))
    print(f"V2 derivado: {len(v2):,} filas; desvío mensual máximo frente a V3: {100*desvio:.2f} %")
    del g, v2
pd.DataFrame(hashes).to_csv(ROOT / "tablas/hashes_datos.csv", index=False)

def csv_de(vol):
    nombre = VOLUMENES[vol][0]
    return FUENTE / nombre / "csv" / f"{nombre}.csv"
for v in VOLUMENES:
    print(v, VOLUMENES[v], csv_de(v).stat().st_size, "bytes")

Sube: ['Dataset_Panaderia_mediano.zip', 'Dataset_Panaderia_grande.zip']


Saving Dataset_Panaderia_pequeno.zip to Dataset_Panaderia_pequeno.zip
Saving Dataset_Panaderia_mediano.zip to Dataset_Panaderia_mediano.zip
Saving Dataset_Panaderia_grande.zip to Dataset_Panaderia_grande.zip
Hashes verificados: 12
V2 derivado: 500,000 filas; desvío mensual máximo frente a V3: 1.88 %
V1 ('mediano', 100000) 6898923 bytes
V2 ('derivado_500k', 500000) 34811783 bytes
V3 ('grande', 1000000) 69622846 bytes


## 4 Calidad de datos y reconciliación

Se evaluó la calidad de los datos de origen mediante el conteo de registros, la unicidad de claves, la identificación de valores nulos y la validación de la expresión `TotalLinea = Cantidad × PrecioUnitario`. También se verificó la correspondencia entre las cabeceras y los detalles de venta, la cobertura de los 61 meses y la paridad entre los archivos CSV y Parquet.

Los resultados de las validaciones se registraron en los archivos `calidad_<volumen>.csv`. En esta etapa no se modificaron ni eliminaron registros; las reglas de rechazo se aplicaron durante el proceso ETL.

In [5]:
ESQUEMA = StructType([
    StructField("IdDetalle", LongType()), StructField("IdVenta", LongType()),
    StructField("FechaVenta", TimestampType()), StructField("IdCliente", LongType()),
    StructField("IdEmpleado", LongType()), StructField("MetodoPago", StringType()),
    StructField("IdProducto", LongType()), StructField("Cantidad", IntegerType()),
    StructField("PrecioUnitario", DoubleType()), StructField("TotalLinea", DoubleType())])
COLS = [f.name for f in ESQUEMA]

def leer_csv_tipado(ruta):
    return spark.read.csv(str(ruta), header=True, schema=ESQUEMA, mode="PERMISSIVE",
                          timestampFormat="yyyy-MM-dd HH:mm:ss")

def leer_csv_texto(ruta):
    return spark.read.csv(str(ruta), header=True, inferSchema=False)

calidad = []
for v, (nombre, n) in VOLUMENES.items():
    d = leer_csv_tipado(csv_de(v))
    a = d.agg(F.count("*").alias("filas"), F.countDistinct("IdDetalle").alias("ids"),
              F.countDistinct("IdVenta").alias("ventas"), F.countDistinct("IdProducto").alias("productos"),
              F.countDistinct("IdCliente").alias("clientes"), F.countDistinct("IdEmpleado").alias("empleados"),
              F.countDistinct(F.date_format("FechaVenta", "yyyy-MM")).alias("meses"),
              F.sum(F.col("Cantidad").isNull().cast("int") + F.col("PrecioUnitario").isNull().cast("int")
                    + F.col("FechaVenta").isNull().cast("int") + F.col("IdProducto").isNull().cast("int")).alias("nulos_clave"),
              F.sum((F.abs(F.col("TotalLinea") - F.col("Cantidad") * F.col("PrecioUnitario")) > 0.005).cast("int")).alias("total_mal"),
              F.min("Cantidad").alias("cant_min"), F.max("Cantidad").alias("cant_max"),
              F.round(F.sum("TotalLinea"), 2).alias("suma_total_linea"),
              F.min("FechaVenta").alias("fecha_min"), F.max("FechaVenta").alias("fecha_max")).first().asDict()
    cab = (d.groupBy("IdVenta").agg(F.countDistinct("FechaVenta").alias("f"), F.countDistinct("IdCliente").alias("c"),
           F.countDistinct("IdEmpleado").alias("e"), F.countDistinct("MetodoPago").alias("m"))
           .filter((F.col("f") > 1) | (F.col("c") > 1) | (F.col("e") > 1) | (F.col("m") > 1)).count())
    fila = dict(volumen=v, origen=nombre, esperado=n, **a, cabeceras_conflictivas=cab)
    pq = FUENTE / nombre / "datos" / nombre
    if pq.exists():
        p = spark.read.parquet(str(pq)).agg(F.count("*").alias("n"), F.round(F.sum("TotalLinea"), 2).alias("s")).first()
        fila.update(parquet_filas=p["n"], parquet_suma=p["s"], paridad_csv_parquet=bool(p["n"] == a["filas"] and abs(p["s"] - a["suma_total_linea"]) < 0.01))
    ref = FUENTE / nombre / f"calidad_{nombre}.csv"
    if ref.exists():
        r = pd.read_csv(ref).iloc[0]
        fila.update(calidad_lineas=int(r["lineas"]), calidad_ventas=int(r["ventas_distintas"]), calidad_productos=int(r["productos_distintos"]))
    assert a["filas"] == n and a["ids"] == n, "Pérdida o repetición de IdDetalle."
    assert a["nulos_clave"] == 0 and a["total_mal"] == 0 and cab == 0
    calidad.append(fila)
calidad = pd.DataFrame(calidad)
calidad.to_csv(ROOT / "tablas/calidad_origen.csv", index=False)
display(calidad.T)

,0,1,2
volumen,V1,V2,V3
origen,mediano,derivado_500k,grande
esperado,100000,500000,1000000
filas,100000,500000,1000000
ids,100000,500000,1000000
ventas,48172,49998,50000
productos,500,500,500
clientes,10000,10000,10000
empleados,100,100,100
meses,61,61,61



## 5. Implementación de las estructuras A (3FN) y B (estrella), consulta Q1 y referencia de exactitud

Q1 devuelve, para cada mes calendario, los 10 productos con mayor SUM(Cantidad); los empates se resuelven por menor IdProducto para que el ranking sea determinista (4.2). En los datos entregados los empates en el décimo lugar son frecuentes, así que este desempate decide la exactitud.

- **Estructura A (3FN):** representa el modelo relacional mediante las tablas `ventas`, `detalle_venta`, `productos`, `clientes` y `empleados`, almacenadas en formato Parquet a partir de los archivos CSV. La consulta Q1 utiliza las relaciones entre `detalle_venta`, `ventas` y `productos`.

- **Estructura B (estrella):** corresponde al Data Mart, compuesto por `dw.fact_ventas` y las dimensiones `dim_fecha`, `dim_producto`, `dim_cliente`, `dim_empleado` y `dim_metodo_pago`, generadas mediante los procesos ETL y ELT de la etapa 6. Para obtener resultados equivalentes, la consulta agrupa por `id_producto` y no por `sk_producto`, debido a que la dimensión de productos conserva versiones históricas mediante SCD tipo 2.

La exactitud de ambas estructuras se compara con una referencia independiente calculada mediante pandas a partir de los archivos CSV. Ninguna estructura recibe índices ni ajustes propios: se escribe Parquet con las mismas reglas (4 archivos en tablas grandes, 1 en dimensiones).


In [6]:
Q1_A = """
WITH m AS (
  SELECT date_format(v.FechaVenta, 'yyyy-MM') AS anio_mes, d.IdProducto AS IdProducto, SUM(d.Cantidad) AS unidades
  FROM detalle_venta d
  JOIN ventas v ON v.IdVenta = d.IdVenta
  JOIN productos p ON p.IdProducto = d.IdProducto
  GROUP BY date_format(v.FechaVenta, 'yyyy-MM'), d.IdProducto),
r AS (
  SELECT anio_mes, IdProducto, unidades,
         ROW_NUMBER() OVER (PARTITION BY anio_mes ORDER BY unidades DESC, IdProducto ASC) AS pos
  FROM m)
SELECT anio_mes, pos, IdProducto, unidades FROM r WHERE pos <= 10 ORDER BY anio_mes, pos
"""
Q1_B = """
WITH m AS (
  SELECT df.anio_mes AS anio_mes, dp.id_producto AS IdProducto, SUM(f.cantidad) AS unidades
  FROM fact_ventas f
  JOIN dim_fecha df ON f.sk_fecha = df.sk_fecha
  JOIN dim_producto dp ON f.sk_producto = dp.sk_producto
  GROUP BY df.anio_mes, dp.id_producto),
r AS (
  SELECT anio_mes, IdProducto, unidades,
         ROW_NUMBER() OVER (PARTITION BY anio_mes ORDER BY unidades DESC, IdProducto ASC) AS pos
  FROM m)
SELECT anio_mes, pos, IdProducto, unidades FROM r WHERE pos <= 10 ORDER BY anio_mes, pos
"""
(ROOT / "consulta_Q1_A.sql").write_text(Q1_A)
(ROOT / "consulta_Q1_B.sql").write_text(Q1_B)

def top10_referencia(csv):
    """Referencia independiente de Spark: pandas sobre el CSV del volumen."""
    df = pd.read_csv(csv, usecols=["FechaVenta", "IdProducto", "Cantidad"], parse_dates=["FechaVenta"])
    df["anio_mes"] = df.FechaVenta.dt.strftime("%Y-%m")
    g = df.groupby(["anio_mes", "IdProducto"], as_index=False).Cantidad.sum().rename(columns={"Cantidad": "unidades"})
    g = g.sort_values(["anio_mes", "unidades", "IdProducto"], ascending=[True, False, True])
    g["pos"] = g.groupby("anio_mes").cumcount() + 1
    return g[g.pos <= TOP_N][["anio_mes", "pos", "IdProducto", "unidades"]].reset_index(drop=True)

def a_dict(filas):
    """Lista de tuplas (anio_mes, pos, IdProducto, unidades) -> {mes: [(pos, id, unidades)]}"""
    out = {}
    for mes, pos, idp, uni in filas:
        out.setdefault(mes, []).append((int(pos), int(idp), int(uni)))
    return {m: sorted(x) for m, x in out.items()}

def exactitud(resultado, ref):
    """% de meses cuyo Top-10 (producto y posición) coincide con la referencia."""
    r = a_dict(resultado)
    meses = set(r) | set(ref)
    ok = sum(1 for m in meses if r.get(m) == ref.get(m))
    return 100.0 * ok / len(meses), ok, len(meses)

def cargar_base_operativa(vol):
    """Estructura A: carga el volumen en tablas normalizadas (3FN)."""
    d = ROOT / "datos" / vol / "A"
    src = leer_csv_tipado(csv_de(vol))
    def w(df, nombre, k=4):
        df.repartition(k).write.mode("overwrite").parquet(str(d / nombre))
    w(src.select("IdVenta", "FechaVenta", "IdCliente", "IdEmpleado", "MetodoPago").dropDuplicates(["IdVenta"]), "ventas")
    w(src.select("IdDetalle", "IdVenta", "IdProducto", "Cantidad", "PrecioUnitario", "TotalLinea"), "detalle_venta")
    w(src.select("IdProducto").distinct(), "productos", 1)
    w(src.select("IdCliente").distinct(), "clientes", 1)
    w(src.select("IdEmpleado").distinct(), "empleados", 1)

def registrar_vistas(estructura, vol):
    base = ROOT / "datos" / vol
    if estructura == "A":
        tablas = {t: base / "A" / t for t in ["ventas", "detalle_venta", "productos", "clientes", "empleados"]}
    else:
        tablas = {t: base / "B_dw" / t for t in ["fact_ventas", "dim_fecha", "dim_producto", "dim_cliente", "dim_empleado", "dim_metodo_pago"]}
    for nombre, ruta in tablas.items():
        spark.read.parquet(str(ruta)).createOrReplaceTempView(nombre)

def bytes_dir(p):
    return sum(f.stat().st_size for f in Path(p).rglob("*.parquet"))
print("Funciones de la etapa 5 listas.")

Funciones de la etapa 5 listas.


## 6 Protocolo de ejecución: EXP-02 (carga) y EXP-01 (consulta) por volumen

Los volúmenes se procesan en el orden de la Tabla 4.7 (V3, V2, V1). En cada volumen: (a) se carga la estructura A; (b) se ejecutan las 10 cargas de EXP-02 en el orden registrado, recreando el data mart desde cero antes de cada una; (c) con el data mart cargado, se ejecutan las 10 mediciones de EXP-01; (d) se repite una vez el procedimiento maestro de cada patrón para comprobar la idempotencia (H4).

**EXP-02.** `ETL` lee el CSV **ya tipado** y escribe `staging`; `ELT` carga primero todo como texto en `raw` y luego convierte dentro del motor con `TRY_CAST`, normaliza coma decimal y vacíos y escribe `staging`. Desde `staging`, ambos reutilizan **los mismos** pasos: reglas RCAL-01 a RCAL-05 (unicidad, completitud, validez, consistencia, integridad), dimensiones, hecho y validación RCAL-06. El tiempo incluye extracción, transformación, carga y validación; no incluye recrear el data mart ni la reconciliación mensual.

**EXP-01.** Por repetición: `clearCache()`, relectura de las tablas, 10 consultas de calentamiento descartadas, y luego 5 sesiones simultáneas de 20 consultas Q1 (100 mediciones). La latencia p95 se calcula por rango más cercano; el throughput es consultas completadas ÷ tiempo de pared. Cada medición cronometra `collect()`, que recupera como máximo 610 filas (61 meses × 10): es seguro aquí y no lo sería con un millón de filas. AQE queda desactivado y no se usa `cache()` de Spark. Existirá caché del sistema operativo y calentamiento de la JVM, y por eso se mide después de calentar.

Se guardan **logs crudos** (JSONL) de cada ejecución y de cada latencia individual: es la evidencia que permite sustentar los resultados en el informe.

In [7]:
MESES_TOTAL = 61
def log(nombre, obj):
    with (ROOT / "logs" / nombre).open("a") as f:
        f.write(json.dumps(obj, default=str) + "\n")

# ---------- Pasos comunes de ETL y ELT ----------
def validar_y_separar(stg):
    """RCAL-01 a RCAL-03: devuelve staging con la columna `motivo` (nula si la fila es válida)."""
    w = Window.partitionBy("IdDetalle").orderBy("IdVenta")
    x = stg.withColumn("_rn", F.row_number().over(w))
    nulo = (F.col("IdDetalle").isNull() | F.col("IdProducto").isNull() | F.col("FechaVenta").isNull()
            | F.col("Cantidad").isNull() | F.col("PrecioUnitario").isNull())
    motivo = (F.when(nulo, F.lit("RCAL-02 completitud"))
               .when(F.col("_rn") > 1, F.lit("RCAL-01 unicidad"))
               .when((F.col("Cantidad") <= 0) | (F.col("PrecioUnitario") < 0) | (F.col("FechaVenta") > F.current_timestamp()),
                     F.lit("RCAL-03 validez")))
    return x.withColumn("motivo", motivo).drop("_rn")

def agregar(df, destino, claves, k=1):
    """Inserta solo claves que aún no existen (idempotencia). Devuelve (filas_antes, filas_nuevas)."""
    destino = Path(destino)
    if destino.exists() and any(destino.glob("*.parquet")):
        previo = spark.read.parquet(str(destino))
        antes = previo.count()
        nuevos = df.join(previo.select(*claves), claves, "left_anti")
    else:
        antes, nuevos = 0, df
    nuevos.repartition(k).write.mode("append").parquet(str(destino))
    despues = spark.read.parquet(str(destino)).count()
    return antes, despues - antes

def cargar_dw(stg, dw):
    x = validar_y_separar(stg).persist(StorageLevel.MEMORY_AND_DISK)
    leidas = x.count()
    rech = x.filter(F.col("motivo").isNotNull())
    rechazadas = rech.count()
    muestra_rech = rech.limit(200).toPandas() if rechazadas else None
    ok = x.filter(F.col("motivo").isNull())
    # RCAL-04: el subtotal se recalcula como cantidad × precio y se cuenta lo corregido.
    ok = (ok.withColumn("subtotal", F.round(F.col("Cantidad") * F.col("PrecioUnitario"), 2))
            .withColumn("id_cliente", F.coalesce("IdCliente", F.lit(-1)))      # miembro SIN-CLIENTE
            .withColumn("id_empleado", F.coalesce("IdEmpleado", F.lit(-1)))
            .withColumn("metodo_pago", F.coalesce("MetodoPago", F.lit("SIN-DATO"))))
    recalculados = ok.filter(F.abs(F.col("TotalLinea") - F.col("subtotal")) > 0.005).count()
    sk_mp = F.pmod(F.xxhash64("metodo_pago"), F.lit(2147483647)).cast("int")
    dim_fecha = (ok.select(F.to_date("FechaVenta").alias("fecha")).distinct()
        .select(F.date_format("fecha", "yyyyMMdd").cast("int").alias("sk_fecha"), "fecha",
                F.year("fecha").alias("anio"), F.month("fecha").alias("mes"),
                F.date_format("fecha", "yyyy-MM").alias("anio_mes"), F.dayofmonth("fecha").alias("dia")))
    dim_producto = (ok.select(F.col("IdProducto").alias("id_producto")).distinct()
        .select((F.col("id_producto") * 10 + 1).alias("sk_producto"), "id_producto",
                F.lit(1).alias("version"), F.lit(True).alias("vigente")))
    dim_cliente = ok.select(F.col("id_cliente").alias("sk_cliente")).distinct()
    dim_empleado = ok.select(F.col("id_empleado").alias("sk_empleado")).distinct()
    dim_mp = ok.select(sk_mp.alias("sk_metodo_pago"), F.col("metodo_pago")).distinct()
    agregar(dim_fecha, dw / "dim_fecha", ["sk_fecha"])
    agregar(dim_producto, dw / "dim_producto", ["sk_producto"])
    agregar(dim_cliente, dw / "dim_cliente", ["sk_cliente"])
    agregar(dim_empleado, dw / "dim_empleado", ["sk_empleado"])
    agregar(dim_mp, dw / "dim_metodo_pago", ["sk_metodo_pago"])
    fact = ok.select(F.col("IdDetalle").alias("id_detalle_venta"),
        F.date_format("FechaVenta", "yyyyMMdd").cast("int").alias("sk_fecha"),
        (F.col("IdProducto") * 10 + 1).alias("sk_producto"),
        F.col("id_cliente").alias("sk_cliente"), F.col("id_empleado").alias("sk_empleado"),
        sk_mp.alias("sk_metodo_pago"), F.col("IdVenta").alias("id_venta"),
        F.col("Cantidad").alias("cantidad"), F.col("PrecioUnitario").alias("precio_unitario"), F.col("subtotal"))
    _, nuevas = agregar(fact, dw / "fact_ventas", ["id_detalle_venta"], k=4)
    validas = leidas - rechazadas
    x.unpersist(blocking=True)
    return dict(leidas=leidas, rechazadas=rechazadas, cargadas=nuevas, existentes=validas - nuevas,
                recalculados=recalculados, pct_rechazo=100.0 * rechazadas / leidas if leidas else 0.0,
                muestra_rech=muestra_rech)

def validar_dw(dw):
    """RCAL-05 (huérfanos), RCAL-06 (columnas personales), unicidad y huella del data mart."""
    f = spark.read.parquet(str(dw / "fact_ventas"))
    dims = {"dim_fecha": "sk_fecha", "dim_producto": "sk_producto", "dim_cliente": "sk_cliente",
            "dim_empleado": "sk_empleado", "dim_metodo_pago": "sk_metodo_pago"}
    j, personales, flags = f, 0, []
    for d, k in dims.items():
        dd = spark.read.parquet(str(dw / d))
        personales += sum(bool(re.search(r"nombre|telefono|celular|correo|email|dni|direccion", c, re.I)) for c in dd.columns)
        j = j.join(dd.select(k).withColumn("_" + k, F.lit(1)), k, "left")
        flags.append(F.col("_" + k).isNull())
    h = j.agg(F.sum(F.when(flags[0] | flags[1] | flags[2] | flags[3] | flags[4], 1).otherwise(0)).alias("huerfanos"),
              F.count("*").alias("filas"), F.countDistinct("id_detalle_venta").alias("ids"),
              F.sum("cantidad").alias("sum_cant"), F.round(F.sum("subtotal"), 2).alias("sum_sub")).first()
    huerf = int(h["huerfanos"] or 0)
    return dict(huerfanos=huerf, cols_personales=personales, duplicados_dw=int(h["filas"] - h["ids"]),
                filas_dw=int(h["filas"]), huella=f"{h['filas']}|{h['sum_cant']}|{h['sum_sub']}",
                validar_ok=bool(huerf == 0 and personales == 0 and h["filas"] == h["ids"]))

def reconciliar(stg_dir, dw):
    """Diferencia de filas e importe por mes entre staging y data mart; debe igualar lo rechazado."""
    x = validar_y_separar(spark.read.parquet(str(stg_dir)))
    mes = F.coalesce(F.date_format("FechaVenta", "yyyy-MM"), F.lit("SIN-FECHA"))
    imp = F.coalesce(F.round(F.col("Cantidad") * F.col("PrecioUnitario"), 2), F.lit(0.0))
    src = x.groupBy(mes.alias("anio_mes")).agg(F.count("*").alias("filas_src"), F.sum(imp).alias("importe_src"))
    rej = (x.filter(F.col("motivo").isNotNull()).groupBy(mes.alias("anio_mes"))
             .agg(F.count("*").alias("filas_rech"), F.sum(imp).alias("importe_rech")))
    fact = spark.read.parquet(str(dw / "fact_ventas")).join(spark.read.parquet(str(dw / "dim_fecha")), "sk_fecha")
    dwm = fact.groupBy("anio_mes").agg(F.count("*").alias("filas_dw"), F.sum("subtotal").alias("importe_dw"))
    r = src.join(rej, "anio_mes", "left").join(dwm, "anio_mes", "left").fillna(0).toPandas()
    r["dif_filas"] = r.filas_src - r.filas_dw
    r["dif_importe"] = r.importe_src - r.importe_dw
    r["conciliado"] = (r.dif_filas == r.filas_rech) & np.isclose(r.dif_importe, r.importe_rech, atol=0.01)
    return r.sort_values("anio_mes").reset_index(drop=True)

# ---------- Procedimientos maestros ----------
def extraer_etl(csv, stg_dir):
    leer_csv_tipado(csv).repartition(4).write.mode("overwrite").parquet(str(stg_dir))

ELT_SQL = """
SELECT TRY_CAST(TRIM(IdDetalle) AS BIGINT) AS IdDetalle,
       TRY_CAST(TRIM(IdVenta) AS BIGINT) AS IdVenta,
       TRY_CAST(TRIM(FechaVenta) AS TIMESTAMP) AS FechaVenta,
       TRY_CAST(TRIM(IdCliente) AS BIGINT) AS IdCliente,
       TRY_CAST(TRIM(IdEmpleado) AS BIGINT) AS IdEmpleado,
       NULLIF(TRIM(MetodoPago), '') AS MetodoPago,
       TRY_CAST(TRIM(IdProducto) AS BIGINT) AS IdProducto,
       TRY_CAST(TRIM(Cantidad) AS INT) AS Cantidad,
       TRY_CAST(REPLACE(NULLIF(TRIM(PrecioUnitario), ''), ',', '.') AS DOUBLE) AS PrecioUnitario,
       TRY_CAST(REPLACE(NULLIF(TRIM(TotalLinea), ''), ',', '.') AS DOUBLE) AS TotalLinea
FROM raw_ventas
"""
(ROOT / "elt_transformar.sql").write_text(ELT_SQL)

def elt_cargar_y_transformar(csv, raw_dir, stg_dir):
    leer_csv_texto(csv).repartition(4).write.mode("overwrite").parquet(str(raw_dir))   # L: todo como texto
    spark.read.parquet(str(raw_dir)).createOrReplaceTempView("raw_ventas")
    spark.sql(ELT_SQL).repartition(4).write.mode("overwrite").parquet(str(stg_dir))      # T: en el motor

def maestro(patron, vol, trat, recrear=True):
    base = ROOT / "datos" / vol
    dw, stg_dir, raw_dir = base / "B_dw", base / f"staging_{patron}", base / "raw_elt"
    if recrear:
        shutil.rmtree(dw, ignore_errors=True)
    inicio = datetime.now(timezone.utc).isoformat()
    t0 = time.perf_counter()
    if patron == "ETL":
        extraer_etl(csv_de(vol), stg_dir)
    else:
        elt_cargar_y_transformar(csv_de(vol), raw_dir, stg_dir)
    res = cargar_dw(spark.read.parquet(str(stg_dir)), dw)
    val = validar_dw(dw)
    dur = time.perf_counter() - t0
    muestra = res.pop("muestra_rech")
    if muestra is not None:
        muestra.to_csv(ROOT / "logs" / f"rechazados_{vol}_{trat}.csv", index=False)
    rec = None
    if recrear:
        rec = reconciliar(stg_dir, dw)
        rec.to_csv(ROOT / "tablas" / f"reconciliacion_{vol}_{trat}.csv", index=False)
    fila = dict(run_id=RUN_ID, volumen=vol, n=VOLUMENES[vol][1], experimento="EXP-02", tratamiento=trat,
                patron=patron, inicio_utc=inicio, duracion_s=dur, throughput_filas_s=res["leidas"] / dur,
                **res, **val, reconciliado=None if rec is None else bool(rec.conciliado.all()),
                meses_conciliados=None if rec is None else int(len(rec)))
    return fila
print("Procedimientos maestros definidos.")

Procedimientos maestros definidos.


In [ ]:
# ---------- Medición de Q1 (EXP-01) ----------
def p95_rango_mas_cercano(x):
    s = np.sort(np.asarray(x))
    return float(s[math.ceil(0.95 * len(s)) - 1])

def medir_Q1(estructura, vol, trat, orden, ref):
    sql = Q1_A if estructura == "A" else Q1_B
    spark.catalog.clearCache()                      # política de caché idéntica en todos los tratamientos
    registrar_vistas(estructura, vol)
    plan = ROOT / "planes" / f"{vol}_{estructura}.txt"
    if not plan.exists():
        plan.write_text(spark.sql("EXPLAIN FORMATTED " + sql).collect()[0][0])
    for _ in range(CALENTAMIENTO):                  # calentamiento descartado
        spark.sql(sql).collect()
    def sesion(_):
        lat, filas = [], None
        for _ in range(CONSULTAS_SESION):
            t = time.perf_counter()
            filas = spark.sql(sql).collect()
            lat.append((time.perf_counter() - t) * 1000.0)
        return lat, [tuple(r) for r in filas]
    inicio = datetime.now(timezone.utc).isoformat()
    t0 = time.perf_counter()
    with ThreadPoolExecutor(SESIONES) as ex:
        res = list(ex.map(sesion, range(SESIONES)))
    pared = time.perf_counter() - t0
    lat = np.array([x for l, _ in res for x in l])
    iguales = all(r[1] == res[0][1] for r in res)
    exa, ok, tot = exactitud(res[0][1], ref)
    fila = dict(run_id=RUN_ID, volumen=vol, n=VOLUMENES[vol][1], experimento="EXP-01", tratamiento=trat,
                estructura=estructura, repeticion=int(trat[1:]), orden=orden, inicio_utc=inicio,
                mediciones=int(lat.size), p95_ms=p95_rango_mas_cercano(lat), mediana_ms=float(np.median(lat)),
                media_ms=float(lat.mean()), max_ms=float(lat.max()), pared_s=pared,
                throughput_cons_s=lat.size / pared, exactitud_pct=exa, meses_ok=ok, meses_total=tot,
                sesiones_coinciden=bool(iguales))
    log("latencias_Q1.jsonl", dict(volumen=vol, tratamiento=trat, latencias_ms=[round(float(x), 3) for x in lat]))
    return fila

# ---------- Ejecución por volumen ----------
for vol in [v for v in ORDEN_VOLUMENES if v in VOLUMENES]:
    f1, f2 = ROOT / "tablas" / f"EXP01_{vol}.csv", ROOT / "tablas" / f"EXP02_{vol}.csv"
    if f1.exists() and f2.exists():
        print(vol, "ya completo en esta ejecución; se omite.")
        continue
    nombre, n = VOLUMENES[vol]
    print(f"=== {vol} ({nombre}, {n:,} líneas) ===")
    cargar_base_operativa(vol)
    ref_df = top10_referencia(csv_de(vol))
    ref_df.to_csv(ROOT / "tablas" / f"referencia_top10_{vol}.csv", index=False)
    ref = a_dict(map(tuple, ref_df.itertuples(index=False, name=None)))
    assert len(ref) == MESES_TOTAL, f"La referencia tiene {len(ref)} meses"

    e2 = []
    for pos, trat in enumerate([t for t in ORDEN_EXP02[vol] if int(t[3:]) <= REPS], 1):
        fila = maestro(trat[:3], vol, trat, recrear=True)
        fila.update(repeticion=int(trat[3:]), orden=pos)
        e2.append(fila); log("ejecuciones_EXP02.jsonl", fila)
        print(f"  EXP-02 {trat}: {fila['duracion_s']:.1f} s | {fila['throughput_filas_s']:,.0f} filas/s | rechazo {fila['pct_rechazo']:.2f} %")
    assert len({f["huella"] for f in e2}) == 1, "ETL y ELT produjeron data marts distintos."

    e1 = []
    for pos, trat in enumerate([t for t in ORDEN_EXP01[vol] if int(t[1:]) <= REPS], 1):
        fila = medir_Q1(trat[0], vol, trat, pos, ref)
        e1.append(fila); log("ejecuciones_EXP01.jsonl", fila)
        print(f"  EXP-01 {trat}: p95 {fila['p95_ms']:.0f} ms | {fila['throughput_cons_s']:.2f} cons/s | exactitud {fila['exactitud_pct']:.1f} %")

    idem = []   # H4: segunda ejecución sobre los mismos datos, sin recrear el data mart
    for patron in ("ETL", "ELT"):
        f = maestro(patron, vol, f"{patron}-2a", recrear=False)
        f.update(experimento="EXP-02-idempotencia", repeticion=0, orden=0)
        idem.append(f); log("ejecuciones_EXP02.jsonl", f)
        print(f"  Idempotencia {patron}: filas nuevas = {f['cargadas']} | existentes = {f['existentes']}")
    pd.DataFrame(idem).to_csv(ROOT / "tablas" / f"EXP02_idempotencia_{vol}.csv", index=False)
    pd.DataFrame([dict(volumen=vol, bytes_A=bytes_dir(ROOT / "datos" / vol / "A"),
                       bytes_B=bytes_dir(ROOT / "datos" / vol / "B_dw"))]).to_csv(ROOT / "tablas" / f"almacenamiento_{vol}.csv", index=False)
    pd.DataFrame(e2).to_csv(f2, index=False)
    pd.DataFrame(e1).to_csv(f1, index=False)
print("Protocolo terminado.")

=== V3 (grande, 1,000,000 líneas) ===
  EXP-02 ETL1: 58.2 s | 17,168 filas/s | rechazo 0.00 %
  EXP-02 ELT2: 47.2 s | 21,169 filas/s | rechazo 0.00 %
  EXP-02 ETL3: 33.1 s | 30,198 filas/s | rechazo 0.00 %
  EXP-02 ETL4: 31.1 s | 32,184 filas/s | rechazo 0.00 %
  EXP-02 ELT1: 40.1 s | 24,925 filas/s | rechazo 0.00 %
  EXP-02 ETL5: 29.0 s | 34,435 filas/s | rechazo 0.00 %
  EXP-02 ELT3: 36.7 s | 27,254 filas/s | rechazo 0.00 %
  EXP-02 ETL2: 28.5 s | 35,089 filas/s | rechazo 0.00 %
  EXP-02 ELT5: 38.0 s | 26,343 filas/s | rechazo 0.00 %
  EXP-02 ELT4: 38.1 s | 26,250 filas/s | rechazo 0.00 %
  EXP-01 A5: p95 11430 ms | 0.59 cons/s | exactitud 100.0 %
  EXP-01 A4: p95 9882 ms | 0.63 cons/s | exactitud 100.0 %
  EXP-01 B3: p95 7293 ms | 0.88 cons/s | exactitud 100.0 %
  EXP-01 B4: p95 7026 ms | 0.90 cons/s | exactitud 100.0 %
  EXP-01 A1: p95 8198 ms | 0.70 cons/s | exactitud 100.0 %
  EXP-01 A3: p95 9543 ms | 0.64 cons/s | exactitud 100.0 %
  EXP-01 B5: p95 6814 ms | 0.95 cons/s | exacti

## 7 Estadística descriptiva y visualizaciones

Reporta mediana, mínimo-máximo, media, desviación estándar y coeficiente de variación (CV) por tratamiento, con las 5 repeticiones. **No selecciones la mejor ejecución**: la tabla consolidada del informe (6.3) debe mostrar unidades, número de repeticiones y variabilidad. Cada punto de una caja es una repetición completa, no una consulta. Si el CV supera 10 %, el tratamiento se marca para repetirse antes de interpretarlo; no repitas hasta obtener la configuración ganadora, documenta cada repetición adicional.

In [ ]:
def leer_consolidado(prefijo):
    partes = [pd.read_csv(p) for p in sorted((ROOT / "tablas").glob(f"{prefijo}_V*.csv")) if "idempotencia" not in p.name and "almacenamiento" not in p.name]
    return pd.concat(partes, ignore_index=True)

e1 = leer_consolidado("EXP01"); e2 = leer_consolidado("EXP02")
e1.to_csv(ROOT / "tablas/EXP01_resultados.csv", index=False)
e2.to_csv(ROOT / "tablas/EXP02_resultados.csv", index=False)
idem = pd.concat([pd.read_csv(p) for p in sorted((ROOT / "tablas").glob("EXP02_idempotencia_V*.csv"))], ignore_index=True)
orden_n = sorted(e1.n.unique())

def resumir(df, grupo, col):
    r = (df.groupby(grupo)[col].agg(repeticiones="count", mediana="median", minimo="min", maximo="max",
                                    media="mean", sd="std").reset_index())
    r["CV_pct"] = 100 * r.sd / r.media
    r["repetir_por_CV"] = r.CV_pct > CV_MAX
    return r.rename(columns={c: f"{c}_{col}" for c in ["repeticiones", "mediana", "minimo", "maximo", "media", "sd", "CV_pct", "repetir_por_CV"]})

res_p95 = resumir(e1, ["volumen", "n", "estructura"], "p95_ms")
res_thr = resumir(e1, ["volumen", "n", "estructura"], "throughput_cons_s")
res_dur = resumir(e2, ["volumen", "n", "patron"], "duracion_s")
res_cargaT = resumir(e2, ["volumen", "n", "patron"], "throughput_filas_s")
for nombre, t in [("resumen_p95", res_p95), ("resumen_throughput_consultas", res_thr),
                  ("resumen_duracion_carga", res_dur), ("resumen_throughput_carga", res_cargaT)]:
    t.to_csv(ROOT / f"tablas/{nombre}.csv", index=False); print(nombre); display(t)

colores = {"A": "tab:red", "B": "tab:blue", "ETL": "tab:green", "ELT": "tab:orange"}
vols = list(VOLUMENES)
fig, axes = plt.subplots(1, len(vols), figsize=(4.6 * len(vols), 4), squeeze=False)
for ax, v in zip(axes[0], vols):
    s = e1[e1.volumen == v]
    ax.boxplot([s[s.estructura == k].p95_ms for k in "AB"], tick_labels=["A (3FN)", "B (estrella)"])
    ax.axhline(UMBRAL_P95_MS, ls="--", color="gray", lw=1)
    ax.set(title=f"{v} ({VOLUMENES[v][1]:,} líneas)", ylabel="p95 (ms)")
fig.suptitle("EXP-01: p95 por repetición. Fuente: elaboración propia, logs/ejecuciones_EXP01.jsonl")
fig.tight_layout(); fig.savefig(ROOT / "figuras/EXP01_cajas.png", dpi=160); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for k in "AB":
    s = res_p95[res_p95.estructura == k].sort_values("n")
    axes[0].errorbar(s.n, s.mediana_p95_ms, yerr=[s.mediana_p95_ms - s.minimo_p95_ms, s.maximo_p95_ms - s.mediana_p95_ms],
                     fmt="o-", capsize=3, color=colores[k], label=k)
axes[0].axhline(UMBRAL_P95_MS, ls="--", color="gray", label="RNF-01: 2 000 ms")
axes[0].set(xscale="log", xlabel="Líneas de venta", ylabel="Mediana de p95 (ms)", title="EXP-01: latencia frente a volumen"); axes[0].legend()
for k in ("ETL", "ELT"):
    s = res_cargaT[res_cargaT.patron == k].sort_values("n")
    axes[1].errorbar(s.n, s.mediana_throughput_filas_s, yerr=[s.mediana_throughput_filas_s - s.minimo_throughput_filas_s, s.maximo_throughput_filas_s - s.mediana_throughput_filas_s],
                     fmt="o-", capsize=3, color=colores[k], label=k)
axes[1].set(xscale="log", xlabel="Líneas de venta", ylabel="Filas/s", title="EXP-02: throughput de carga"); axes[1].legend()
fig.tight_layout(); fig.savefig(ROOT / "figuras/escala_EXP01_EXP02.png", dpi=160); plt.show()

fig, axes = plt.subplots(1, len(vols), figsize=(4.6 * len(vols), 3.6), squeeze=False)
for ax, v in zip(axes[0], vols):
    for k in "AB":
        s = e1[(e1.volumen == v) & (e1.estructura == k)].sort_values("orden")
        ax.plot(s.orden, s.p95_ms, "o", color=colores[k], label=k)
    ax.set(title=v, xlabel="Orden de ejecución (Tabla 4.7)", ylabel="p95 (ms)"); ax.legend()
fig.tight_layout(); fig.savefig(ROOT / "figuras/EXP01_orden.png", dpi=160); plt.show()

## 8. Reglas de decisión y comparación estadística

La evaluación se realizó mediante las reglas descriptivas establecidas previamente en la sección 4.1, considerando medianas, rangos de variación, exactitud y umbrales definidos. Como análisis complementario, se aplicó la prueba de Mann-Whitney de una cola por volumen, con corrección de Holm para las tres comparaciones.

Con cinco repeticiones por tratamiento, el valor p mínimo posible es 1/252 ≈ 0,004. La significancia estadística se interpreta junto con la magnitud práctica del efecto y no sustituye los criterios de decisión establecidos. No se calcularon intervalos de confianza debido al reducido número de repeticiones.

**Supuestos y limitaciones.** Las repeticiones se consideran intercambiables bajo H0 dentro de cada volumen. Aunque el orden aleatorizado reduce posibles sesgos, no garantiza independencia debido al uso de la misma JVM y la caché del sistema operativo. Los resultados corresponden a un único equipo y no se generalizan directamente a otros entornos ni a SQL Server.


In [ ]:
def holm(p):
    p = np.asarray(p, float); orden = np.argsort(p); aj = np.zeros(len(p)); acum = 0.0
    for j, i in enumerate(orden):
        acum = max(acum, (len(p) - j) * p[i]); aj[i] = min(1.0, acum)
    return aj

filas, pvals = [], []
for v in sorted(e1.volumen.unique(), key=lambda x: VOLUMENES[x][1]):
    a = e1[(e1.volumen == v) & (e1.estructura == "A")]; b = e1[(e1.volumen == v) & (e1.estructura == "B")]
    ma, mb = a.p95_ms.median(), b.p95_ms.median()
    p = stats.mannwhitneyu(a.p95_ms, b.p95_ms, alternative="greater", method="auto").pvalue
    pvals.append(p)
    filas.append(dict(volumen=v, n=VOLUMENES[v][1], mediana_p95_A=ma, mediana_p95_B=mb,
        reduccion_pct=100 * (1 - mb / ma), brecha_ms=ma - mb,
        c1_reduccion_ge_30=bool(1 - mb / ma >= UMBRAL_REDUCCION),
        c2_rangos_sin_solape=bool(b.p95_ms.max() < a.p95_ms.min()),
        c3_exactitud_100=bool(e1[e1.volumen == v].exactitud_pct.min() == 100.0),
        c4_p95_B_le_2000=bool(mb <= UMBRAL_P95_MS), p_una_cola=p))
dec1 = pd.DataFrame(filas)
dec1["p_Holm"] = holm(pvals)
ultimo = dec1.n.idxmax()
dec1["cumple_volumen"] = dec1.c1_reduccion_ge_30 & dec1.c2_rangos_sin_solape & dec1.c3_exactitud_100 & (dec1.c4_p95_B_le_2000 | (dec1.index != ultimo))
brechas = dec1.sort_values("n").brecha_ms.to_numpy()
crece = bool(len(brechas) > 1 and np.all(np.diff(brechas) > 0))
dec1.to_csv(ROOT / "tablas/decision_EXP01.csv", index=False); display(dec1.T)
print("Brecha absoluta A-B creciente con el volumen (H1, segunda parte):", crece)
print("B mejor que A según la regla de 4.1 (cumple en todos los volúmenes y p95 <= 2 000 ms en el mayor):", bool(dec1.cumple_volumen.all() and crece))

# ---- EXP-02 ----
filas = []
for k in ("ETL", "ELT"):
    s = e2[e2.patron == k]
    med = s.groupby("n").throughput_filas_s.median().sort_index()
    razon = float(med.iloc[-1] / med.iloc[0]) if len(med) > 1 else float("nan")
    idk = idem[idem.patron == k]
    filas.append(dict(patron=k, throughput_mediano_menor_n=float(med.iloc[0]), throughput_mediano_mayor_n=float(med.iloc[-1]),
        razon_mayor_menor=razon, c1_throughput_ge_70=bool(razon >= UMBRAL_THROUGHPUT) if not np.isnan(razon) else None,
        c2_validacion_y_reconciliacion=bool(s.validar_ok.all() and s.reconciliado.all()),
        c3_rechazo_mediano_lt_1=bool(s.groupby("n").pct_rechazo.median().max() < UMBRAL_RECHAZO_PCT),
        c4_segunda_ejecucion_0_nuevas=bool((idk.cargadas == 0).all())))
dec2 = pd.DataFrame(filas)
dec2["escalable"] = dec2[[c for c in dec2.columns if c.startswith("c")]].all(axis=1)
dec2.to_csv(ROOT / "tablas/decision_EXP02.csv", index=False); display(dec2.T)

filas, pv = [], []
for v in sorted(e2.volumen.unique(), key=lambda x: VOLUMENES[x][1]):
    a = e2[(e2.volumen == v) & (e2.patron == "ETL")].throughput_filas_s; b = e2[(e2.volumen == v) & (e2.patron == "ELT")].throughput_filas_s
    p = stats.mannwhitneyu(a, b, alternative="two-sided", method="auto").pvalue; pv.append(p)
    filas.append(dict(volumen=v, mediana_ETL=a.median(), mediana_ELT=b.median(), razon_ETL_sobre_ELT=a.median() / b.median(), p_dos_colas=p))
cmp2 = pd.DataFrame(filas); cmp2["p_Holm"] = holm(pv)
cmp2.to_csv(ROOT / "tablas/comparacion_ETL_ELT.csv", index=False); display(cmp2)


## 9. EXP-03: Comparativo integrado en V3 y análisis de resultados

EXP-03 integra los resultados obtenidos en V3 de EXP-01 y EXP-02, sin realizar nuevas ejecuciones. Los resultados se contrastan con los criterios establecidos en la sección 4.1 y los requisitos no funcionales RNF-01 (p95 ≤ 2 000 ms), RNF-04 (exactitud del 100 %), RNF-06 (idempotencia y reconciliación) y RNF-07 (rechazo inferior al 1 %).

El análisis considera el cumplimiento de estos criterios y delimita las conclusiones a las condiciones observadas en el entorno experimental, sin generalizarlas a otros equipos o plataformas.


In [ ]:
mayor = max(VOLUMENES, key=lambda v: VOLUMENES[v][1])
s1 = e1[e1.volumen == mayor]; s2 = e2[e2.volumen == mayor]
filas = []
for k in "AB":
    s = s1[s1.estructura == k]
    filas.append(dict(escenario=f"EXP-01 estructura {k}", volumen=mayor, mediana=s.p95_ms.median(), minimo=s.p95_ms.min(), maximo=s.p95_ms.max(),
        unidad="p95 (ms)", CV_pct=100 * s.p95_ms.std() / s.p95_ms.mean(), exactitud_pct=s.exactitud_pct.min(),
        RNF_01_cumple=bool(s.p95_ms.median() <= UMBRAL_P95_MS), RNF_04_cumple=bool(s.exactitud_pct.min() == 100.0)))
for k in ("ETL", "ELT"):
    s = s2[s2.patron == k]; ik = idem[(idem.patron == k) & (idem.volumen == mayor)]
    filas.append(dict(escenario=f"EXP-02 patrón {k}", volumen=mayor, mediana=s.throughput_filas_s.median(), minimo=s.throughput_filas_s.min(), maximo=s.throughput_filas_s.max(),
        unidad="filas/s", CV_pct=100 * s.throughput_filas_s.std() / s.throughput_filas_s.mean(), exactitud_pct=None,
        RNF_06_cumple=bool(s.reconciliado.all() and (ik.cargadas == 0).all()), RNF_07_cumple=bool(s.pct_rechazo.median() < UMBRAL_RECHAZO_PCT)))
exp03 = pd.DataFrame(filas); exp03.to_csv(ROOT / "tablas/EXP03_comparativo.csv", index=False); display(exp03)
alm = pd.concat([pd.read_csv(p) for p in sorted((ROOT / "tablas").glob("almacenamiento_V*.csv"))], ignore_index=True)
display(alm)
print("Cobertura de ejecuciones:", len(e1), "de EXP-01 y", len(e2), "de EXP-02 (diseño completo: 30 y 30).")

### Análisis e interpretación de EXP-03

El análisis integrado considera los siguientes aspectos:

- **Rendimiento:** reducción porcentual de la latencia p95 de la estructura B respecto de A en V3, utilizando como denominador la latencia p95 de A. También se verifica el cumplimiento de RNF-01 (p95 ≤ 2 000 ms) en ambas estructuras.
- **Comportamiento de H1:** se examina si la estructura en estrella presenta la mejora esperada y se identifican posibles factores técnicos, como las uniones con difusión, que puedan explicar las diferencias observadas.
- **Almacenamiento:** se comparan los valores de `bytes_A` y `bytes_B` en cada volumen para identificar la diferencia de espacio ocupado por ambas estructuras.
- **Representatividad de los datos:** se consideran las limitaciones de los volúmenes derivados de 150 000 líneas reales y 500 productos, especialmente para inferir preferencias de compra y patrones de estacionalidad de la panadería. También se analiza la estabilidad del Top-10 entre volúmenes.
- **Aplicación de Q1:** se relacionan los resultados de la consulta con la decisión del propietario sobre la planificación de producción y compras, considerando la exactitud requerida.
- **Alcance de las inferencias:** se diferencia entre correlación y causalidad, considerando las limitaciones del experimento: ejecución en un solo equipo, diferencias tecnológicas respecto de SQL Server, influencia de la caché del sistema operativo y cinco repeticiones por tratamiento.

### Cómo se integraron los resultados en el informe

| Sección del informe | Evidencias utilizadas |
|---|---|
| 4.1, 4.2, 4.3 y 4.7 (Diseño) | Etapa 0 (ficha y cambios declarados), `consulta_Q1_*.sql`, `elt_transformar.sql` |
| 4.4 y 4.6 (Entorno y procedimiento) | `entorno.json`, captura del panel Recursos, `hashes_datos.csv`, Tabla 4.7 usada |
| 6.1 Validación del dataset | `calidad_origen.csv`, `reconciliacion_*.csv` |
| 6.2 Registro de ejecuciones | `logs/ejecuciones_EXP01.jsonl`, `logs/ejecuciones_EXP02.jsonl`, `logs/latencias_Q1.jsonl`, incidentes y desviaciones |
| 6.3 Tabla consolidada | `EXP01_resultados.csv`, `resumen_*.csv`, `decision_EXP01.csv`, `decision_EXP02.csv`, `EXP03_comparativo.csv` |
| 6.4 Visualizaciones | `figuras/*.png` con título, ejes, unidades, fuente y código de evidencia |
| 6.5 Análisis | Respuestas de las etapas 7, 8 y 9 |
| Anexos C, D y E | C: fichas y hashes de datos. D: este notebook `.ipynb` con salidas y los `.sql`. E: logs, tablas, figuras y capturas de planes |


## 10. Conclusiones y recomendaciones

Se sintetizaron los resultados de la experimentación según los objetivos establecidos, considerando las métricas obtenidas y los archivos de evidencia generados. Las conclusiones se sustentan en los resultados medidos y las recomendaciones se orientan a mejorar los procesos evaluados.

| **Objetivo evaluado** | **Resultado numérico y evidencia** | **Conclusión técnica** | **Recomendación** |
|---|---|---|---|
| Calidad y conservación de ventas (etapa 4) | Registros validados, rechazados y reconciliados; `calidad_origen.csv` y `reconciliacion_*.csv`. | Evaluación de la calidad, integridad y conservación de los datos de ventas. | Fortalecer las validaciones de calidad según los errores identificados. |
| EXP-01: estructura 3FN frente a estrella | Latencia p95, throughput, exactitud y almacenamiento; `EXP01_resultados.csv`. | Comparación del rendimiento de ambas estructuras en los tres volúmenes. | Seleccionar la estructura según los resultados de rendimiento y exactitud obtenidos. |
| EXP-02: carga ETL frente a ELT | Duración, throughput, registros rechazados e idempotencia; `logs/ejecuciones_EXP02.jsonl`. | Comparación del comportamiento y escalabilidad de ambos patrones de carga. | Optimizar el patrón que presente mejores resultados y cumpla los criterios de calidad. |
| EXP-03: cumplimiento de RNF-01, RNF-04, RNF-06 y RNF-07 en V3 | Indicadores de latencia, exactitud, idempotencia, reconciliación y rechazo; `EXP03_comparativo.csv`. | Determinación del cumplimiento de los requisitos no funcionales evaluados. | Aplicar mejoras sobre los requisitos que no alcancen los umbrales establecidos. |

Las recomendaciones consideran el cambio técnico propuesto, la evidencia que lo sustenta, los recursos necesarios para su implementación y el indicador que permitirá comprobar su mejora. Las conclusiones se limitan a las condiciones del entorno experimental y distinguen los resultados medidos de las posibles mejoras que requieren una evaluación adicional.


## 11. Cierre y exportación de evidencias

Ejecuta al final. Spark se detiene para cerrar sus logs; para seguir experimentando debes comenzar una nueva ejecución desde la etapa 2. Descarga también este `.ipynb` **con salidas** desde Archivo → Descargar → Descargar .ipynb; el ZIP no captura automáticamente el notebook abierto. Adjunta informe, capturas del panel de recursos y de los planes, contribuciones y declaración de IA.

In [ ]:
spark.stop()
manifest = []
for archivo in sorted(ROOT.rglob("*")):
    if archivo.is_file() and "spark_events" not in archivo.parts and "datos" not in archivo.relative_to(ROOT).parts[:1]:
        manifest.append(dict(ruta=str(archivo.relative_to(ROOT)), bytes=archivo.stat().st_size, sha256=sha256(archivo)))
(ROOT / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
# Los datos intermedios (carpeta datos/) no se incluyen en el ZIP; los hashes de la fuente están en tablas/hashes_datos.csv.
paquete = BASE / ("paquete_" + ROOT.name)
paquete.mkdir(exist_ok=True)
for sub in ["tablas", "figuras", "planes", "logs"]:
    shutil.copytree(ROOT / sub, paquete / sub, dirs_exist_ok=True, ignore=shutil.ignore_patterns("spark_events"))
for f in ["entorno.json", "manifest.json", "consulta_Q1_A.sql", "consulta_Q1_B.sql", "elt_transformar.sql"]:
    shutil.copy(ROOT / f, paquete / f)
zip_path = shutil.make_archive(str(paquete), "zip", root_dir=paquete.parent, base_dir=paquete.name)
print("Archivo generado:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Entorno local: copia el ZIP desde la ruta indicada.")

## Fuentes técnicas

- Apache Spark 4.0.1, instalación: https://spark.apache.org/docs/4.0.1/api/python/getting_started/install.html
- Spark SQL, particiones y AQE: https://spark.apache.org/docs/4.0.1/sql-performance-tuning.html
- Spark SQL, ventanas y `try_cast`: https://spark.apache.org/docs/4.0.1/sql-ref-functions.html
- SciPy Mann-Whitney: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.mannwhitneyu.html
- LEEME y manifest de los paquetes `Dataset_Panaderia_*.zip` (origen, diccionario y SHA-256 de los datos).